In [1]:

# Install required packages
!pip install transformers torch requests Pillow

# Imports
import torch
from transformers import CLIPProcessor, CLIPModel
import torch.nn as nn
from PIL import Image
import requests
from io import BytesIO
import os
import json
import logging

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 34.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlink

In [2]:

# Define CLIP-based classifier
class CLIPClassifier(nn.Module):
    def __init__(self, clip_model_name="openai/clip-vit-base-patch32", num_classes=2):
        super(CLIPClassifier, self).__init__()
        self.clip = CLIPModel.from_pretrained(clip_model_name)
        for param in self.clip.parameters():
            param.requires_grad = False
        hidden_size = self.clip.config.projection_dim
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes)
        )

    def forward(self, input_ids=None, attention_mask=None, pixel_values=None, return_loss=False, labels=None):
        outputs = self.clip(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values)
        text_embeds = outputs.text_embeds
        logits = self.classifier(text_embeds)
        loss = None
        if return_loss and labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels)
        return logits, loss


In [4]:

# Load model and processor
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = CLIPClassifier()
model.load_state_dict(torch.load("/content/clip_fake_news_classifier1.pth", map_location=torch.device("cpu")))
model.eval()


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

CLIPClassifier(
  (clip): CLIPModel(
    (text_model): CLIPTextTransformer(
      (embeddings): CLIPTextEmbeddings(
        (token_embedding): Embedding(49408, 512)
        (position_embedding): Embedding(77, 512)
      )
      (encoder): CLIPEncoder(
        (layers): ModuleList(
          (0-11): 12 x CLIPEncoderLayer(
            (self_attn): CLIPSdpaAttention(
              (k_proj): Linear(in_features=512, out_features=512, bias=True)
              (v_proj): Linear(in_features=512, out_features=512, bias=True)
              (q_proj): Linear(in_features=512, out_features=512, bias=True)
              (out_proj): Linear(in_features=512, out_features=512, bias=True)
            )
            (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
            (mlp): CLIPMLP(
              (activation_fn): QuickGELUActivation()
              (fc1): Linear(in_features=512, out_features=2048, bias=True)
              (fc2): Linear(in_features=2048, out_features=512, bias=Tru

In [5]:

def predict_clip(claim_text, image_url=None):
    if image_url:
        try:
            response = requests.get(image_url)
            image = Image.open(BytesIO(response.content)).convert("RGB")
        except Exception as e:
            logger.warning(f"Failed to load image, using blank. Reason: {e}")
            image = Image.new("RGB", (224, 224), color="white")
    else:
        image = Image.new("RGB", (224, 224), color="white")

    inputs = processor(text=claim_text, images=image, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        logits, _ = model(**inputs)
        pred = torch.argmax(logits, dim=1).item()
    return "Real" if pred == 1 else "Fake"


In [6]:

def fact_check_google(claim_text, api_key):
    url = f"https://factchecktools.googleapis.com/v1alpha1/claims:search?query={claim_text}&key={api_key}"
    response = requests.get(url)
    if response.status_code != 200:
        logger.warning("Failed to fetch from Google Fact Check API")
        return None

    results = response.json().get("claims", [])
    if not results:
        return "No match found"

    top_claim = results[0]
    claim_text = top_claim.get("text", "N/A")
    claim_rating = top_claim.get("claimReview", [{}])[0].get("textualRating", "Unrated")
    return f"{claim_text} --> {claim_rating}"


In [7]:

def unified_fake_news_detection(claim_text, image_url=None, api_key=""):
    print(f"📝 Claim: {claim_text}")

    # CLIP model prediction
    clip_result = predict_clip(claim_text, image_url)
    print(f"🤖 CLIP Model Prediction: {clip_result}")

    # Google Fact Check
    if api_key:
        fact_check_result = fact_check_google(claim_text, api_key)
        print(f"🔍 Google Fact Check: {fact_check_result}")
    else:
        fact_check_result = "API key not provided"
        print("⚠️ Google API skipped (no key)")

    # Summary
    print("\n🧾 Final Verdict Summary:")
    print(f" - Model: {clip_result}")
    print(f" - Fact Check: {fact_check_result}")


In [9]:

def fact_check_google_with_links(claim_text, api_key):
    url = f"https://factchecktools.googleapis.com/v1alpha1/claims:search?query={claim_text}&key={api_key}"
    response = requests.get(url)
    if response.status_code != 200:
        logger.warning("Failed to fetch from Google Fact Check API")
        return None, None

    results = response.json().get("claims", [])
    if not results:
        return "No match found", None

    top_claim = results[0]
    claim_text = top_claim.get("text", "N/A")
    review = top_claim.get("claimReview", [{}])[0]
    claim_rating = review.get("textualRating", "Unrated")
    source_url = review.get("url", None)
    return f"{claim_text} --> {claim_rating}", source_url


In [10]:

def unified_fake_news_detection(claim_text=None, image_url=None, api_key=""):
    print("🧠 Unified Fake News Detection")

    # Validate input
    if not claim_text and not image_url:
        print("❌ Please provide at least a claim text or an image.")
        return

    # Prediction
    clip_result = predict_clip(claim_text or "", image_url)
    print(f"🤖 CLIP Model Prediction: {clip_result}")

    # Google Fact Check
    if api_key and claim_text:
        fact_check_result, source_url = fact_check_google_with_links(claim_text, api_key)
        print(f"🔍 Google Fact Check: {fact_check_result}")
        if source_url:
            print(f"🔗 Source: {source_url}")
    else:
        print("⚠️ Google API skipped (missing key or claim text)")


In [11]:

# Example use (Replace with your API key)
api_key = "#####ur api"
claim = "In 2024, “China made $1 trillion off trade with the United States.”"
image_url = "https://static.politifact.com/CACHE/images/politifact/photos/AP24067198471019/97c6f09abba347e1ed9a451b68ffc6ae.jpg"  # or provide a real URL
unified_fake_news_detection(claim, image_url, api_key)


🧠 Unified Fake News Detection
🤖 CLIP Model Prediction: Real
🔍 Google Fact Check: In 2024, “China made $1 trillion off trade with the United States.” --> False
🔗 Source: https://www.politifact.com/factchecks/2025/apr/11/donald-trump/donald-trump-wrong-china-1-trillion-trade-deficit/
